<a href="https://colab.research.google.com/github/paolavaldes0107-netizen/IA-1/blob/main/Actividad_IA_1_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import pandas as pd
import numpy as np

# cargar dataset
df = pd.read_csv("cleaned_dataset.csv")

# seleccionar 40 muestras aleatorias
muestras = df.sample(n=40, random_state=42)

# dividir en entrenamiento y prueba
train = muestras.iloc[:20]
test = muestras.iloc[20:]

print("Datos de entrenamiento:", train.shape)
print("Datos de prueba:", test.shape)

Datos de entrenamiento: (20, 9)
Datos de prueba: (20, 9)


In [15]:
import math

def distancia_euclidiana(x, y):
    suma = 0
    for i in range(len(x)):
        suma += (x[i] - y[i])**2
    return math.sqrt(suma)

# ejemplos dados
vector1 = [1,106,70,28,135,34.2,0.142,22]
vector2 = [2,102,86,36,120,45.5,0.127,23]

dist = distancia_euclidiana(vector1, vector2)

print("Distancia euclidiana:", dist)

Distancia euclidiana: 26.280985997484947


In [16]:
def knn_predict(train, test_point, k=3):

    distancias = []

    for i in range(len(train)):
        fila = train.iloc[i]
        x_train = fila[:-1].values
        y_train = fila[-1]

        distancia = distancia_euclidiana(x_train, test_point)

        distancias.append((distancia, y_train))

    distancias.sort(key=lambda x: x[0])

    vecinos = distancias[:k]

    votos = [vecino[1] for vecino in vecinos]

    prediccion = max(set(votos), key=votos.count)

    return prediccion


# probar con 10 muestras
resultados = []

for i in range(10):

    test_point = test.iloc[i][:-1].values
    real = test.iloc[i][-1]

    pred = knn_predict(train, test_point, 3)

    resultados.append([real, pred])

tabla = pd.DataFrame(resultados, columns=["Real","Predicho"])
print(tabla)

   Real  Predicho
0   1.0       0.0
1   0.0       0.0
2   1.0       1.0
3   1.0       1.0
4   1.0       1.0
5   1.0       1.0
6   0.0       0.0
7   1.0       0.0
8   0.0       0.0
9   0.0       0.0


/tmp/ipykernel_606/2805312867.py:31: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  real = test.iloc[i][-1]
/tmp/ipykernel_606/2805312867.py:8: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  y_train = fila[-1]


In [17]:
from sklearn.model_selection import train_test_split

X = df.drop("Outcome", axis=1)
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(313, 8)
(79, 8)


In [18]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

knn = KNeighborsClassifier(n_neighbors=3, metric="euclidean")

knn.fit(X_train, y_train)

pred = knn.predict(X_test)

accuracy_crudo = accuracy_score(y_test, pred)

print("Accuracy datos crudos:", accuracy_crudo)

Accuracy datos crudos: 0.810126582278481


In [19]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train_norm = scaler.fit_transform(X_train)
X_test_norm = scaler.transform(X_test)

knn_norm = KNeighborsClassifier(n_neighbors=3)

knn_norm.fit(X_train_norm, y_train)

pred_norm = knn_norm.predict(X_test_norm)

accuracy_norm = accuracy_score(y_test, pred_norm)

print("Accuracy normalizado:", accuracy_norm)

Accuracy normalizado: 0.7341772151898734


In [20]:
from sklearn.preprocessing import StandardScaler

scaler2 = StandardScaler()

X_train_std = scaler2.fit_transform(X_train)
X_test_std = scaler2.transform(X_test)

knn_std = KNeighborsClassifier(n_neighbors=3)

knn_std.fit(X_train_std, y_train)

pred_std = knn_std.predict(X_test_std)

accuracy_std = accuracy_score(y_test, pred_std)

print("Accuracy estandarizado:", accuracy_std)

Accuracy estandarizado: 0.7468354430379747


In [21]:
tabla = pd.DataFrame({
    "Metodo":[
        "KNN sin escalar",
        "KNN normalizado",
        "KNN estandarizado"
    ],
    "Accuracy":[
        accuracy_crudo,
        accuracy_norm,
        accuracy_std
    ]
})

print(tabla)

              Metodo  Accuracy
0    KNN sin escalar  0.810127
1    KNN normalizado  0.734177
2  KNN estandarizado  0.746835


In [22]:
print(df.head())
print(df.shape)

   Pregnancies  Glucose  Blood Pressure  Skin Thickness  Insulin   BMI  \
0            0      129             110              46      130  67.1   
1            0      180              78              63       14  59.4   
2            3      123             100              35      240  57.3   
3            1       88              30              42       99  55.0   
4            0      162              76              56      100  53.2   

   Diabetes Pedigree Function  Age  Outcome  
0                       0.319   26        1  
1                       2.420   25        1  
2                       0.880   22        0  
3                       0.496   26        1  
4                       0.759   25        1  
(392, 9)


#Preguntas
#¿Por qué es importante normalizar o estandarizar los datos antes de usar KNN?

Es importante porque el algoritmo KNN funciona calculando distancias entre los datos. Si una variable tiene valores mucho más grandes que otras, puede influir más en el cálculo de la distancia. Al normalizar o estandarizar los datos, todas las variables quedan en una escala similar y el algoritmo puede comparar los puntos de forma más justa.

#¿Qué diferencias observaste en el accuracy entre los datos crudos, normalizados y estandarizados?

En mi experimento, el modelo con datos sin escalar obtuvo el accuracy más alto (aproximadamente 0.81). Los modelos con datos normalizados y estandarizados también funcionaron bien, pero con un accuracy un poco menor. Esto puede ocurrir porque algunas variables del dataset ya tienen escalas parecidas, por lo que el escalado no cambió mucho el resultado.

#Si aumentamos el valor de k (número de vecinos), ¿cómo crees que cambiaría el rendimiento del modelo?

Si aumentamos el valor de k, el modelo toma en cuenta más vecinos para hacer la predicción. Esto puede hacer que el modelo sea más estable y menos sensible a valores atípicos. Sin embargo, si k es demasiado grande, el modelo puede perder precisión porque estaría considerando demasiados puntos que tal vez no son tan similares.

#¿Qué ventaja tiene implementar KNN manualmente antes de usar scikit-learn?

Implementar KNN manualmente ayuda a entender mejor cómo funciona el algoritmo. Permite ver paso a paso cómo se calcula la distancia entre los datos, cómo se seleccionan los vecinos más cercanos y cómo se decide la clase final. Después de entender esto, usar librerías como scikit-learn resulta más fácil.

#¿Qué limitaciones presenta KNN cuando se aplica a conjuntos de datos grandes o con muchas dimensiones?

Una de las principales limitaciones es que puede ser lento cuando el conjunto de datos es muy grande, ya que necesita calcular la distancia con muchos puntos. Además, cuando hay muchas dimensiones o variables, las distancias entre los puntos pueden volverse menos significativas, lo que puede afectar el rendimiento del modelo.